# 02 - Brand Specification Generator

##  Objective
This notebook demonstrates how to transform a raw, unstructured brand brief into a structured **Brand Specification** using a Large Language Model (LLM).

The Brand Specification is the **foundation** of the entire BRANDORA pipeline. Every subsequent step (name generation, slogan creation, color palette selection, logo generation) depends on this structured data.

---

##  What This Notebook Does

1. **Loads** a sample brand brief from `test_briefs.json`
2. **Connects** to Groq API (using secure environment variables)
3. **Sends** the brief to an LLM with a carefully crafted prompt
4. **Forces** the LLM to output structured JSON (not free-form text)
5. **Validates** the output to ensure it matches the expected schema

---

##  Key Technical Decisions

### Why Groq?
- Free tier with generous rate limits
- Fast inference (Llama 3.3 70B runs in <1 second)
- Supports JSON mode for structured outputs

### Why JSON Mode?
Without JSON mode, the LLM might output:

In [2]:
import os
import json
from openai import OpenAI
from kaggle_secrets import UserSecretsClient

# 1. قراءة المفاتيح السرية من Kaggle Secrets
user_secrets = UserSecretsClient()
openrouter_key = user_secrets.get_secret("OPENROUTER_API_KEY")
inception_key = user_secrets.get_secret("INCEPTION_API_KEY")

# 2. تهيئة عميل OpenRouter (للنماذج الرخيصة)
openrouter_client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=openrouter_key,
)

# 3. تهيئة عميل Inception Labs (لنموذج Mercury)
inception_client = OpenAI(
    base_url="https://api.inceptionlabs.ai/v1",
    api_key=inception_key,
)

print("✅ تم الاتصال بالمنصتين بنجاح!")
print("📡 OpenRouter: جاهز للنماذج الرخيصة")
print("📡 Inception Labs: جاهز لـ Mercury")

✅ تم الاتصال بالمنصتين بنجاح!
📡 OpenRouter: جاهز للنماذج الرخيصة
📡 Inception Labs: جاهز لـ Mercury


In [ ]:


try:
    response = inception_client.chat.completions.create(
        model="mercury-2.5",
        messages=[
            {"role": "user", "content": "Say hello in one sentence."}
        ],
        max_tokens=500,
        reasoning_effort="low"
    )
    
    content = response.choices[0].message.content
    
    if content:
        print(f"✅ نجح Mercury! الرد: '{content.strip()}'")
        print(f"\n📊 إحصائيات الرد:")
        print(f"   - finish_reason: {response.choices[0].finish_reason}")
        if hasattr(response, 'usage') and response.usage:
            print(f"   - prompt_tokens: {response.usage.prompt_tokens}")
            print(f"   - completion_tokens: {response.usage.completion_tokens}")
            print(f"   - total_tokens: {response.usage.total_tokens}")
    else:
        print("⚠️ المحتوى لا يزال فارغاً")
        print(f"   - finish_reason: {response.choices[0].finish_reason}")
        if hasattr(response, 'usage') and response.usage:
            print(f"   - total_tokens المستهلكة: {response.usage.total_tokens}")
    
except Exception as e:
    print(f"❌ فشل! السبب: {str(e)[:200]}")

In [3]:
pip install -U bitsandbytes>=0.46.1

Note: you may need to restart the kernel to use updated packages.


In [4]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch
import traceback
import re
import gc

print(f"GPU: {torch.cuda.get_device_name(0)}")

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
)

models_to_check = [
    {"name": "Qwen/Qwen3-8B", "model_id": "Qwen/Qwen3-8B"},
    {"name": "LiquidAI/LFM2.5-2.6B", "model_id": "LiquidAI/LFM2.5-2.6B"},
]

results = {}  # عشان نخزن الردود ونقارنها بعدين

print("🔍 بدء اختبار النماذج محلياً على GPU...\n")

for model_info in models_to_check:
    print(f"⏳ جاري تحميل: {model_info['name']}")
    model = None  # نعرّفه هون عشان الـ finally يلاقيه حتى لو فشل التحميل بنفسه
    try:
        tokenizer = AutoTokenizer.from_pretrained(model_info["model_id"])
        model = AutoModelForCausalLM.from_pretrained(
            model_info["model_id"],
            quantization_config=quant_config,
            device_map="auto"
        )

        messages = [{"role": "user", "content": "Generate a brand name and a short tagline for a modern coffee shop targeting university students. Reply in this format: Name: ... | Tagline: ..."}]

        try:
            inputs = tokenizer.apply_chat_template(
                messages, add_generation_prompt=True,
                return_tensors="pt", return_dict=True,
                enable_thinking=False
            ).to(model.device)
        except TypeError:
            inputs = tokenizer.apply_chat_template(
                messages, add_generation_prompt=True,
                return_tensors="pt", return_dict=True
            ).to(model.device)

        outputs = model.generate(**inputs, max_new_tokens=300)
        input_length = inputs["input_ids"].shape[-1]
        content = tokenizer.decode(outputs[0][input_length:], skip_special_tokens=True)

        clean_content = re.sub(r'.*</think>\s*', '', content, flags=re.DOTALL).strip()
        results[model_info["name"]] = clean_content
        print(f"   ✅ الرد: '{clean_content}'\n")

    except Exception as e:
        print(f"   ❌ فشل! نوع الخطأ: {type(e).__name__}")
        traceback.print_exc()
        results[model_info["name"]] = None
        print()

    finally:
        # هاد الجزء بينفّذ دايماً — نجح الموديل ولا فشل
        if model is not None:
            del model
        gc.collect()
        torch.cuda.empty_cache()

print("✅ انتهى الاختبار!\n")
print("📋 ملخص النتائج:")
for name, result in results.items():
    print(f"\n{name}:\n{result}")

GPU: Tesla T4
🔍 بدء اختبار النماذج محلياً على GPU...

⏳ جاري تحميل: Qwen/Qwen3-8B


Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

   ✅ الرد: 'Name: BrewBud | Tagline: Fuel Your Study Sessions, One Cup at a Time.'

⏳ جاري تحميل: LiquidAI/LFM2.5-2.6B


Loading weights:   0%|          | 0/266 [00:00<?, ?it/s]

   ✅ الرد: 'The user wants me to generate a brand name and a short tagline for a modern coffee shop targeting university students. The response must be in the specific format: "Name: ... | Tagline: ..."

I need to create:
1. A catchy, modern brand name suitable for a university student coffee shop
2. A concise tagline that appeals to students

Brainstorming names:
- "Campus Brew" - classic but effective
- "Study & Sip" - directly mentions studying and drinking
- "The Lecture Latte" - plays on academic lectures
- "Caffeine Campus" - straightforward
- "Student Bean" - simple and direct
- "Perk Up" - energetic, relates to energy/studying
- "The Study Spot" - functional
- "Mug & Mind" - combines coffee with thinking
- "Academic Roast" - punny, academic reference
- "Late Night Latte" - targets night owls (common among students)
- "The Library Cup" - references library study

For a modern vibe, I want something trendy, memorable, and appealing to Gen Z/Millennial students. Names like "Perk U